# Colab Environment Probe — closes spec gaps E1-E4 (NO training, ~2 min)
Run All on a GPU runtime (Runtime -> Change runtime type -> GPU; pick the biggest tier offers).
Then paste the full output back. This measures whether this Colab session can hold a 31B QLoRA.
It does NOT download the 31B weights (that's gated + large) — it reports VRAM, deps, and HF access.

In [ ]:
# E1 - GPU model + VRAM actually granted this session
import subprocess, torch
print('=== E1: GPU / VRAM ===')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free,driver_version','--format=csv'], capture_output=True, text=True).stdout)
print('torch sees CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'GPU0: {torch.cuda.get_device_name(0)}  total={total/1e9:.1f}GB  free={free/1e9:.1f}GB')
    VRAM_GB = total/1e9
    print('VERDICT E1:', 'OK for 31B QLoRA (>=40GB)' if VRAM_GB>=40 else ('MARGINAL 24-40GB' if VRAM_GB>=24 else 'TOO SMALL for 31B (<24GB) -> need A100-class'))
else:
    print('NO GPU — set Runtime -> GPU and re-run')

In [ ]:
# E4 - pinned deps that work with Gemma-4 QLoRA on Colab; record the resolved versions
import subprocess, sys
print('=== E4: install + versions (quiet) ===')
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
                'transformers>=4.49','peft>=0.14','bitsandbytes>=0.45','accelerate>=1.3','safetensors'], check=False)
import torch, transformers, peft, bitsandbytes, accelerate
for m in (torch, transformers, peft, bitsandbytes, accelerate):
    print(f'{m.__name__:14s} {m.__version__}')
print('RECORD these versions in the spec E4 (the working combo).')

In [ ]:
# E2 - HF access to the bf16 31B base (set your token in Colab Secrets as HF_TOKEN, or paste below).
# This checks ACCESS + size WITHOUT downloading the full weights (metadata only).
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')  # fallback
print('HF token present:', bool(HF_TOKEN))
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
# CANDIDATES: confirm the exact bf16 31B sibling id. Try the likely ones; report which are reachable.
CANDIDATES = ['google/gemma-4-31b-it', 'google/gemma-4-27b-it', 'google/gemma-4-12b-it']
for mid in CANDIDATES:
    try:
        info = api.model_info(mid, files_metadata=True)
        gb = sum((f.size or 0) for f in info.siblings)/1e9
        print(f'OK    {mid:28s} gated={getattr(info,"gated","?")} approx_repo_GB={gb:.1f}')
    except Exception as e:
        print(f'NO    {mid:28s} {type(e).__name__}: {str(e)[:90]}')
print('VERDICT E2: pick the largest reachable id that fits E1 VRAM in 4-bit (~weights/2 for NF4).')

In [ ]:
# E3 - 4-bit load footprint smoke on a SMALL Gemma-4 (e2b) to confirm the NF4 path works here.
# (We do NOT 4-bit-load the 31B in the probe; this proves the pipeline + measures per-GB overhead.)
import torch, time
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
MID = 'google/gemma-4-e2b-it'  # smallest Gemma-4; proves NF4 load path on THIS Colab
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
f0,_ = torch.cuda.mem_get_info(); t=time.time()
try:
    m = AutoModelForCausalLM.from_pretrained(MID, quantization_config=bnb, device_map={'':0}, attn_implementation='eager')
    f1,_ = torch.cuda.mem_get_info()
    print(f'E3: NF4 load of {MID} OK in {time.time()-t:.0f}s; used ~{(f0-f1)/1e9:.1f}GB VRAM')
    print('VERDICT E3: NF4 pipeline WORKS on this Colab. Extrapolate: 31B 4-bit ~= 16-18GB weights + headroom.')
    del m; torch.cuda.empty_cache()
except Exception as e:
    print('E3 FAILED:', type(e).__name__, str(e)[:200])

## After Run All
Paste the full output back. We will fill spec E1-E4, then decide D3/D4 (which bf16 base, whether this GPU holds 31B) before building any training data or running the full job.